# MRAM Calibration Notebook

This notebook documents threshold models, drift detection, and test data for the spintronics MRAM MVP.

## Overview

The MRAM (Magnetoresistive Random Access Memory) MVP implements threshold-based state detection using tunnel magnetoresistance (TMR). This notebook provides:

1. **Threshold Model Calibration** - Determining optimal resistance thresholds
2. **Drift Detection** - Monitoring threshold stability over time
3. **Test Data Generation** - Creating synthetic test cases for validation

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Import spintronics modules
import sys
sys.path.insert(0, '..')
from spintronics import hash_packet, compute_receipt_hash
from spintronics.verifiers import verify_mram_write_read, decode_resistance_to_bit

## 1. Threshold Model Calibration

### Theory

MRAM uses TMR effect where resistance depends on magnetic alignment:
- **Parallel state (P)**: Low resistance R_P → bit 0
- **Antiparallel state (AP)**: High resistance R_AP → bit 1

TMR ratio: TMR = (R_AP - R_P) / R_P

Typical values:
- R_P ≈ 2 kΩ
- R_AP ≈ 5 kΩ
- TMR ≈ 1.5 (150%)

In [ ]:
# Define calibration parameters
R_P_nominal = 2000.0  # Ohms (parallel state)
R_AP_nominal = 5000.0  # Ohms (antiparallel state)
TMR_nominal = (R_AP_nominal - R_P_nominal) / R_P_nominal

print(f"Nominal TMR ratio: {TMR_nominal:.2f}")
print(f"Nominal switching margin: {R_AP_nominal - R_P_nominal:.0f} Ω")

# Compute optimal threshold (midpoint)
R_threshold = (R_P_nominal + R_AP_nominal) / 2.0
print(f"Optimal threshold: {R_threshold:.0f} Ω")

### Resistance Distribution Simulation

Simulate realistic resistance distributions with Gaussian noise:

In [ ]:
# Simulation parameters
n_samples = 1000
noise_sigma = 200.0  # Ohms (5% of switching margin)

# Generate distributions
np.random.seed(42)
R_P_distribution = np.random.normal(R_P_nominal, noise_sigma, n_samples)
R_AP_distribution = np.random.normal(R_AP_nominal, noise_sigma, n_samples)

# Plot distributions
plt.figure(figsize=(10, 6))
plt.hist(R_P_distribution, bins=50, alpha=0.6, label='Parallel (bit 0)', color='blue')
plt.hist(R_AP_distribution, bins=50, alpha=0.6, label='Antiparallel (bit 1)', color='red')
plt.axvline(R_threshold, color='green', linestyle='--', linewidth=2, label='Threshold')
plt.xlabel('Resistance (Ω)')
plt.ylabel('Count')
plt.title('MRAM Resistance State Distributions')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Compute error rates
P_errors = np.sum(R_P_distribution > R_threshold) / n_samples
AP_errors = np.sum(R_AP_distribution < R_threshold) / n_samples
total_error = (P_errors + AP_errors) / 2.0

print(f"\nError rates:")
print(f"  P state misread as AP: {P_errors*100:.2f}%")
print(f"  AP state misread as P: {AP_errors*100:.2f}%")
print(f"  Overall bit error rate: {total_error*100:.2f}%")

## 2. Drift Detection

Monitor threshold stability over time and temperature:

In [ ]:
# Simulate drift over time
time_points = np.linspace(0, 100, 50)  # Time in hours
drift_rate = 0.5  # Ohms per hour

# Simulate R_P and R_AP drift (opposite directions)
R_P_drift = R_P_nominal + drift_rate * time_points + np.random.normal(0, 50, len(time_points))
R_AP_drift = R_AP_nominal - drift_rate * time_points + np.random.normal(0, 50, len(time_points))

# Compute margin over time
margin_drift = R_AP_drift - R_P_drift

plt.figure(figsize=(10, 6))
plt.plot(time_points, R_P_drift, 'b-', label='R_P (parallel)')
plt.plot(time_points, R_AP_drift, 'r-', label='R_AP (antiparallel)')
plt.plot(time_points, (R_P_drift + R_AP_drift)/2, 'g--', label='Threshold (midpoint)')
plt.xlabel('Time (hours)')
plt.ylabel('Resistance (Ω)')
plt.title('MRAM Resistance Drift Over Time')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(time_points, margin_drift, 'purple')
plt.axhline(R_AP_nominal - R_P_nominal, color='green', linestyle='--', label='Nominal margin')
plt.xlabel('Time (hours)')
plt.ylabel('Switching Margin (Ω)')
plt.title('MRAM Switching Margin Drift')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Initial margin: {margin_drift[0]:.0f} Ω")
print(f"Final margin: {margin_drift[-1]:.0f} Ω")
print(f"Margin degradation: {margin_drift[0] - margin_drift[-1]:.0f} Ω ({(margin_drift[0] - margin_drift[-1])/margin_drift[0]*100:.1f}%)")

## 3. Test Data Generation

Generate synthetic test receipts for validation:

In [ ]:
def generate_test_receipt(data_bit, resistance_state=None, address=None):
    """Generate a test MRAM receipt."""
    
    # Auto-generate resistance based on bit if not provided
    if resistance_state is None:
        if data_bit == 0:
            resistance_state = np.random.normal(R_P_nominal, noise_sigma)
        else:
            resistance_state = np.random.normal(R_AP_nominal, noise_sigma)
    
    # Generate address if not provided
    if address is None:
        address = f"0x{np.random.randint(0, 0xFFFF):04X}"
    
    # Create receipt
    receipt = {
        "schema_version": "mram_write_read_v1",
        "receipt_id": "0" * 64,  # Placeholder
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "write_operation": {
            "address": address,
            "data_bit": data_bit,
            "pulse_program": {
                "amplitude": 1.8,
                "duration": 10.0,
                "pulse_shape": "rectangular"
            },
            "write_voltage": 1.8,
            "write_current": 150.0
        },
        "read_operation": {
            "resistance_state": resistance_state,
            "read_voltage": 0.5,
            "read_current": 0.5 * 1e6 / resistance_state,  # I = V/R
            "decoded_bit": 0 if resistance_state < R_threshold else 1
        },
        "threshold_verification": {
            "low_resistance_threshold": R_P_nominal,
            "high_resistance_threshold": R_AP_nominal,
            "switching_margin": R_AP_nominal - R_P_nominal,
            "verification_passed": True,
            "tmr_ratio": TMR_nominal
        }
    }
    
    # Compute receipt hash
    receipt_hash = compute_receipt_hash(
        receipt["write_operation"],
        receipt["read_operation"]
    )
    receipt["receipt_id"] = receipt_hash
    
    return receipt

# Generate test cases
print("Generating test receipts...\n")

# Test case 1: Write 0, read 0
receipt_0 = generate_test_receipt(0)
is_valid, details = verify_mram_write_read(receipt_0)
print(f"Test 1 - Write 0: Valid={is_valid}, R={details['resistance_state']:.0f}Ω, Bit={details['decoded_bit']}")

# Test case 2: Write 1, read 1
receipt_1 = generate_test_receipt(1)
is_valid, details = verify_mram_write_read(receipt_1)
print(f"Test 2 - Write 1: Valid={is_valid}, R={details['resistance_state']:.0f}Ω, Bit={details['decoded_bit']}")

# Save test receipts
with open('../examples/test_receipt_write0.json', 'w') as f:
    json.dump(receipt_0, f, indent=2)

with open('../examples/test_receipt_write1.json', 'w') as f:
    json.dump(receipt_1, f, indent=2)

print("\nTest receipts saved to examples/")

## 4. Threshold Optimization

Determine optimal threshold to minimize bit error rate:

In [ ]:
# Sweep threshold values
thresholds = np.linspace(R_P_nominal - 500, R_AP_nominal + 500, 100)
error_rates = []

for threshold in thresholds:
    P_errors = np.sum(R_P_distribution > threshold) / n_samples
    AP_errors = np.sum(R_AP_distribution < threshold) / n_samples
    total_error = (P_errors + AP_errors) / 2.0
    error_rates.append(total_error)

# Find optimal threshold
optimal_idx = np.argmin(error_rates)
optimal_threshold = thresholds[optimal_idx]
optimal_error = error_rates[optimal_idx]

plt.figure(figsize=(10, 6))
plt.plot(thresholds, np.array(error_rates) * 100, 'b-', linewidth=2)
plt.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal ({optimal_threshold:.0f} Ω)')
plt.axvline(R_threshold, color='green', linestyle='--', linewidth=2, label=f'Midpoint ({R_threshold:.0f} Ω)')
plt.xlabel('Threshold (Ω)')
plt.ylabel('Bit Error Rate (%)')
plt.title('Threshold Optimization')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Optimal threshold: {optimal_threshold:.0f} Ω")
print(f"Optimal error rate: {optimal_error*100:.2f}%")
print(f"Midpoint threshold: {R_threshold:.0f} Ω")
print(f"Midpoint error rate: {error_rates[np.argmin(np.abs(thresholds - R_threshold))]*100:.2f}%")

## 5. Summary and Recommendations

### Calibrated Parameters

- **Low resistance threshold (R_P)**: 2000 Ω
- **High resistance threshold (R_AP)**: 5000 Ω
- **Switching margin**: 3000 Ω
- **Optimal read threshold**: ~3500 Ω (midpoint)
- **TMR ratio**: 1.5 (150%)

### Recommendations

1. **Use midpoint threshold** for symmetric error distribution
2. **Monitor margin drift** - recalibrate if margin drops below 2000 Ω
3. **Temperature compensation** - adjust thresholds for thermal drift
4. **Error correction** - Implement ECC for BER > 1e-6
5. **Periodic calibration** - Re-measure distributions every 24 hours

### Next Steps

- Validate with real hardware measurements
- Implement adaptive threshold tracking
- Characterize temperature dependence
- Develop calibration automation